<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-19-April-7-2026/Lecture-19_DimensionalityReduction-NeuralTSNE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 19 - Dimensionality Reduction 3 - NeuralTSNE


Import all basic pacakges

In [ ]:
!pip install neuraltsne

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Alanine Dipeptide

Here we consider a alanine dipeptide in vacuum that is simple model often used to test methods. It consists of one alanine (ala) residues and capping group. It is conformational dynamics can be understood in terms of the backbond dihedral angles $\Phi$ and $\Psi$.

![Alanine Dipeptide](https://github.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-16-March-24-2026/Ala1.png?raw=true)

We will consider dataset from molecular dynamics simulations, in particular parallel tempering (PT) simulations, that should give correct equilibrium sampling according to the Boltzmann distribution.

The dataset we consider is at different temperatures, 416 K and 576 K. It is from a 100 ns PT simulations where we save variables every 10 ps, so we have 10000 samples.

The variables we consider are the dihedrals, $\Phi$ and $\Psi$, and also the set of all heavy atom distances.

In [ ]:
# Download datasets

%%bash
datasets="
AlanineDipeptide_T-416K_Dihedrals.data
AlanineDipeptide_T-416K_HeavyAtomDistances.data
"

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Lecture-16-March-24-2026/Datasets"

for d in ${datasets}
do
  wget ${url}/${d} &> /dev/null
done

ls

First we consider the dihedral angles

In [ ]:
!head AlanineDipeptide_T-416K_Dihedrals.data

In [ ]:
def get_variables_names_from_header(filename):
  with open(filename, 'r') as f:
    header = f.readline()
    variables = header.split()[2:]
  return variables

In [ ]:
dih_variables = get_variables_names_from_header("AlanineDipeptide_T-416K_Dihedrals.data")

data_dih = pd.read_csv("AlanineDipeptide_T-416K_Dihedrals.data", header=None, names=dih_variables, sep='\\s+', comment="#")

# only take every N-th value
stride = 1
data_dih = data_dih.iloc[::stride]
# this is to reset the index
data_dih.reset_index(drop=True, inplace=True)

In [ ]:
data_dih

In [ ]:
x_label = 'phi'
y_label = 'psi'

t = data_dih['time']
x = data_dih[x_label]
y = data_dih[y_label]


plt.plot(t, x, '.', label='x')
plt.xlabel('time')
plt.ylabel(x_label)
plt.show()

plt.plot(t, y, '.', label='y')
plt.xlabel('time')
plt.ylabel(y_label)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_scatter, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sc = ax_scatter.scatter(x, y, s=14, alpha=0.8, linewidths=0)
ax_scatter.set_xlabel(x_label)
ax_scatter.set_ylabel(y_label)
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()

g = sns.JointGrid(height=7, ratio=5)
ax_joint, ax_top, ax_right = g.ax_joint, g.ax_marg_x, g.ax_marg_y
sns.kdeplot(x=x, y=y,
            ax=ax_joint,
            bw_adjust=0.6,
            fill=True,
            levels=8)
ax_joint.set_xlabel(x_label)
ax_joint.set_ylabel(y_label)
sns.kdeplot(x=x, ax=ax_top, fill=True, color="#a855f7", bw_adjust=0.6)
sns.kdeplot(y=y, ax=ax_right, fill=True, color="#f97316", bw_adjust=0.6)
plt.show()



In [ ]:
phi = data_dih['phi'].to_numpy()

data_dih['C7eq_basin'] = np.where( (phi > 0) & (phi < 2) ,0 , 1)

import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(['red', 'blue'])
plt.scatter(data_dih['phi'], data_dih['psi'],c=data_dih['C7eq_basin'],s=10, cmap=cmap)
cbar = plt.colorbar(ticks=[0.25,0.75])
cbar.ax.set_yticklabels(['C7ax', 'C7_eq'])
plt.show()

In [ ]:
!head AlanineDipeptide_T-416K_HeavyAtomDistances.data

In [ ]:
dist_variables = get_variables_names_from_header("AlanineDipeptide_T-416K_HeavyAtomDistances.data")

In [ ]:
dist_variables

In [ ]:
data_dist = pd.read_csv("AlanineDipeptide_T-416K_HeavyAtomDistances.data", header=None, names=dist_variables, sep='\\s+', comment="#")

# only take every N-th value
stride = 1
data_dist = data_dist.iloc[::stride]
# this is to reset the index
data_dist.reset_index(drop=True, inplace=True)

## NeuralTSNE

Here we are going to use the [NeuralTSNE code](https://pubs.acs.org/doi/10.1021/acs.jcim.5c01107) that implements t-SNE using a neural network.

Based on [example](https://github.com/NeuralTSNE/NeuralTSNE/blob/main/examples/example-adp.ipynb) from the NeuralTSNE code.

In [ ]:
import torch
import NeuralTSNE as ntsne

from torch.utils.data import TensorDataset, random_split
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning import Trainer, seed_everything

torch.set_float32_matmul_precision("high")

In [ ]:
# Check if CUDA is available
print(torch.cuda.is_available())  # True = GPU available

# See which device you're on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # 'cuda' or 'cpu'

# Get GPU name (if available)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # e.g. 'NVIDIA GeForce RTX 3090'
    print(torch.cuda.device_count())      # Number of GPUs

In [ ]:
distances = data_dist.drop(columns=['time']).to_numpy()

X = torch.tensor(distances, dtype=torch.float32)
dataset = TensorDataset(X)

valid = 0.2
n = len(dataset)
n_train = int((1.0 - valid) * n)
n_valid = n - n_train
train_dataset, valid_dataset = random_split(dataset, [n_train, n_valid])

In [ ]:
nn_multipliers = [2, 2, 2]
model = ntsne.TSNE.NeuralNetwork.NeuralNetwork(initial_features=45, n_components=2, multipliers=nn_multipliers)

In [ ]:
model

In [ ]:
param_tsne = ntsne.TSNE.ParametricTSNE.ParametricTSNE(
    loss_fn="kl_divergence",
    perplexity=50,
    batch_size=500,
    early_exaggeration_epochs=0,
    early_exaggeration_value=0,
    max_iterations=500,
    model=model
)

In [ ]:
seed_everything(seed=42, workers=True)

early_stopping = EarlyStopping(monitor="val_loss", min_delta=1e-6, patience=10)

trainer = Trainer(
    enable_progress_bar=False,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    log_every_n_steps=1,
    max_epochs=param_tsne.max_iterations,
    callbacks=[early_stopping],
    default_root_dir=".",
    deterministic=True # seed
)

In [ ]:
reducer = ntsne.TSNE.Modules.DimensionalityReduction(
    tsne=param_tsne,
    shuffle=False,
    optimizer="adam",
    lr=1e-3
)

In [ ]:
train_dataloader, _ = param_tsne.create_dataloaders(train_dataset, None)
valid_dataloader, _ = param_tsne.create_dataloaders(valid_dataset, None)

full_dataloader, _ = param_tsne.create_dataloaders(dataset, None)

In [ ]:
trainer.fit(reducer, train_dataloaders=train_dataloader, val_dataloaders=[valid_dataloader])



In [ ]:
Z = trainer.predict(reducer, full_dataloader)
Z = np.concatenate(Z, axis=0)

In [ ]:
Z

In [ ]:
plt.scatter(Z[:, 0], Z[:, 1], c=data_dih['C7eq_basin'])
plt.show()